# 6. Random Forest

**Machine Learning Fundamentals and Predictive Analytics — Notebook 6 of 11**

A single decision tree is unstable: change 10% of the rows and you get a different tree
(Notebook 3). A **random forest** turns that weakness into a strength — grow hundreds of
deliberately different trees and average them. The individual errors are partly independent, so
they cancel.

The result is the strongest general-purpose algorithm for tabular data that requires almost no
tuning. If you have a table and a target and one afternoon, fit a random forest.

### What you will learn

1. **Bagging**: bootstrap aggregating, and why averaging reduces variance
2. The **two sources of randomness** in a random forest, and why decorrelation matters
3. **Out-of-bag (OOB)** scoring — free validation
4. The hyperparameters that actually matter
5. **Feature importance**: impurity, permutation, and their biases
6. **Extra Trees** and **gradient boosting** — the neighbours
7. Random forests for **regression**
8. Handling **imbalanced** classes
9. Strengths, weaknesses, and a full worked comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              BaggingClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, HistGradientBoostingClassifier,
                              HistGradientBoostingRegressor)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.model_selection import (train_test_split, cross_val_score, GridSearchCV,
                                     RandomizedSearchCV, StratifiedKFold, KFold,
                                     validation_curve, learning_curve)
from sklearn.metrics import (accuracy_score, roc_auc_score, classification_report,
                             confusion_matrix, mean_squared_error, r2_score,
                             average_precision_score, recall_score, precision_score)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from scipy.stats import randint, uniform
import time

rng = np.random.default_rng(seed=6)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
SKF = StratifiedKFold(5, shuffle=True, random_state=0)
CV = KFold(5, shuffle=True, random_state=0)

---
## 6.1 Bagging: the core idea

**Bootstrap AGGregatING**:

1. Draw $B$ **bootstrap samples** — each of size $n$, sampled **with replacement** from the
   training set
2. Fit one model on each
3. Average the predictions (regression) or take a majority vote (classification)

Why it works. If you average $B$ predictors each with variance $\sigma^2$ and pairwise
correlation $\rho$:

$$\operatorname{Var}(\text{average}) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$$

As $B \to \infty$ the second term vanishes and you are left with $\rho\sigma^2$. So:

- **Averaging helps most when the models are uncorrelated** ($\rho$ small)
- **More models never hurt**, but returns diminish once $\frac{1-\rho}{B}\sigma^2$ is small
- The floor is $\rho\sigma^2$ — which is why **reducing $\rho$ is the real lever**

Note what bagging does *not* change: the **bias**. Each bootstrap tree has roughly the same
bias as a tree on the full data, and so does their average.

In [ ]:
# Bootstrap sampling: what a bootstrap sample looks like
original = np.arange(10)
for i in range(4):
    boot = rng.choice(original, size=10, replace=True)
    oob = sorted(set(original) - set(boot))
    print(f"  bootstrap {i}: {sorted(boot)}   out-of-bag: {oob}")

print(f"\nExpected fraction of rows LEFT OUT of a bootstrap sample:")
print(f"  (1 - 1/n)^n -> 1/e = {1/np.e:.4f}, i.e. about 36.8%")
for n_ in (10, 100, 1000, 100_000):
    print(f"  n = {n_:>7,}: {(1 - 1/n_)**n_:.4f}")
print("\nThose left-out rows are the OOB set, and they are free validation data.")

In [ ]:
# The variance-reduction formula, verified
def measure_variance(estimator_factory, n_boot=200, n_train=120, seed=0):
    '''Variance of predictions at fixed test points across independent training sets.'''
    g = np.random.default_rng(seed)
    X_test = g.normal(size=(200, 4))
    preds = []
    for _ in range(n_boot):
        Xt = g.normal(size=(n_train, 4))
        yt = (Xt[:, 0] + Xt[:, 1]**2 - Xt[:, 2] + g.normal(0, 0.4, n_train) > 0).astype(int)
        preds.append(estimator_factory().fit(Xt, yt).predict_proba(X_test)[:, 1])
    P = np.array(preds)
    return P.var(axis=0).mean()

rows = []
for name, fac in [("1 tree", lambda: DecisionTreeClassifier(random_state=0)),
                  ("bagging, 10 trees",
                   lambda: BaggingClassifier(DecisionTreeClassifier(), n_estimators=10,
                                             random_state=0)),
                  ("bagging, 50 trees",
                   lambda: BaggingClassifier(DecisionTreeClassifier(), n_estimators=50,
                                             random_state=0)),
                  ("random forest, 50 trees",
                   lambda: RandomForestClassifier(n_estimators=50, random_state=0))]:
    rows.append({"model": name, "prediction_variance": measure_variance(fac, n_boot=60)})
v = pd.DataFrame(rows)
v["reduction_vs_single"] = (1 - v.prediction_variance / v.prediction_variance.iloc[0]).round(3)
print(v.round(5).to_string(index=False))
print("\nBagging cuts the variance sharply. The random forest cuts it further, because")
print("it also DECORRELATES the trees -- which is the topic of the next section.")

---
## 6.2 What makes a forest "random"

A random forest is bagging plus one crucial extra ingredient:

1. **Bootstrap sampling of rows** (bagging) — each tree sees a different ~63% of the data
2. **Random subset of features at every split** (`max_features`) — each split considers only
   $m$ of the $p$ features

The second is the innovation (Breiman, 2001). Without it, one dominant feature would be chosen
as the root of nearly every tree, making them highly correlated — and correlated trees do not
average away.

**Defaults for `max_features`:**

| Task | Default | Rationale |
|---|---|---|
| Classification | $\sqrt{p}$ | Strong decorrelation |
| Regression | $p$ (all features) in scikit-learn, though $p/3$ is the classic recommendation | Regression trees need more features to find good splits |

There is a trade-off: smaller `max_features` means less correlation (good) but weaker individual
trees (bad).

In [ ]:
# Does max_features really decorrelate the trees? Measure it.
m = 1_200
Xf = rng.normal(size=(m, 10))
# one dominant feature, several weak ones
zf = 3.0*Xf[:, 0] + 0.6*Xf[:, 1] + 0.5*Xf[:, 2] + 0.4*Xf[:, 3] + rng.normal(0, 1, m)
yf = (zf > 0).astype(int)
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(Xf, yf, test_size=0.3, random_state=0,
                                              stratify=yf)

print(f"{'max_features':>14}{'root feature = x0':>20}{'mean tree corr':>17}{'CV AUC':>10}")
for mf in [None, 8, 5, 3, 1]:
    forest = RandomForestClassifier(n_estimators=60, max_features=mf,
                                    random_state=0).fit(Xf_tr, yf_tr)
    roots = [t.tree_.feature[0] for t in forest.estimators_]
    frac_x0 = np.mean(np.array(roots) == 0)
    # correlation between the trees' predictions on the test set
    P = np.array([t.predict_proba(Xf_te)[:, 1] for t in forest.estimators_])
    corr = np.corrcoef(P)
    mean_corr = (corr.sum() - len(corr)) / (len(corr)**2 - len(corr))
    auc = cross_val_score(RandomForestClassifier(n_estimators=60, max_features=mf,
                                                 random_state=0),
                          Xf_tr, yf_tr, cv=SKF, scoring="roc_auc").mean()
    label = "all" if mf is None else str(mf)
    print(f"{label:>14}{frac_x0:>20.3f}{mean_corr:>17.4f}{auc:>10.4f}")
print("\nWith all features available, x0 is the root of nearly every tree and the trees are")
print("highly correlated. Restricting max_features breaks that up. The best AUC sits at an")
print("intermediate value: enough decorrelation, without crippling each tree.")

In [ ]:
# How many trees? More is (almost) always better, with diminishing returns
counts = [1, 2, 5, 10, 25, 50, 100, 200, 400, 800]
scores, oob = [], []
for B in counts:
    f = RandomForestClassifier(n_estimators=B, oob_score=True, random_state=0,
                               bootstrap=True).fit(Xf_tr, yf_tr)
    scores.append(roc_auc_score(yf_te, f.predict_proba(Xf_te)[:, 1]))
    oob.append(f.oob_score_)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(counts, scores, "o-", color="steelblue")
ax[0].set_xscale("log"); ax[0].set_xlabel("number of trees"); ax[0].set_ylabel("test ROC-AUC")
ax[0].set_title("More trees: monotone improvement, then a plateau")
ax[1].plot(counts, oob, "o-", color="seagreen")
ax[1].set_xscale("log"); ax[1].set_xlabel("number of trees"); ax[1].set_ylabel("OOB accuracy")
ax[1].set_title("OOB score stabilises too")
plt.tight_layout(); plt.show()

print(f"{'trees':>7}{'test AUC':>11}{'OOB acc':>10}")
for B, s, o in zip(counts, scores, oob):
    print(f"{B:>7}{s:>11.4f}{o:>10.4f}")
print("\nUnlike depth or k, n_estimators is not a bias-variance knob you can get wrong.")
print("More trees never overfits -- it only costs time and memory. Use 100-500 and")
print("spend your tuning budget elsewhere.")

---
## 6.3 Out-of-bag scoring: free validation

Each tree is trained on ~63% of the rows, so ~37% are **out-of-bag** for it. Predict each row
using only the trees that did *not* see it, and you have an honest estimate of generalisation —
without a separate validation set and without cross-validation.

Set `oob_score=True`. Caveats: it needs `bootstrap=True`, it is noisy with few trees, and it
assumes the rows are independent (so it is invalid for grouped or time-series data — the same
caution as statistics Notebook 10).

In [ ]:
f_oob = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=0).fit(
    Xf_tr, yf_tr)
cv_acc = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=0),
                         Xf_tr, yf_tr, cv=SKF, scoring="accuracy")

print(f"Training accuracy      : {f_oob.score(Xf_tr, yf_tr):.4f}  <- always near 1.0")
print(f"OOB accuracy           : {f_oob.oob_score_:.4f}  <- free, honest")
print(f"5-fold CV accuracy     : {cv_acc.mean():.4f} +/- {cv_acc.std():.4f}")
print(f"Held-out test accuracy : {f_oob.score(Xf_te, yf_te):.4f}")
print("\nOOB tracks CV closely at a fraction of the cost -- one fit instead of five.")
print("Use it for quick hyperparameter sweeps, then confirm the winner with CV.")

In [ ]:
# Using OOB to sweep a hyperparameter cheaply
print(f"{'min_samples_leaf':>18}{'OOB accuracy':>15}{'CV accuracy':>14}")
for leaf in (1, 2, 5, 10, 20, 50):
    f_ = RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf,
                                oob_score=True, random_state=0).fit(Xf_tr, yf_tr)
    cvv = cross_val_score(RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf,
                                                 random_state=0),
                          Xf_tr, yf_tr, cv=SKF).mean()
    print(f"{leaf:>18}{f_.oob_score_:>15.4f}{cvv:>14.4f}")
print("\nThe two columns rank the options the same way, which is all a sweep needs.")

---
## 6.4 The hyperparameters that matter

| Parameter | What it does | Sensible range | Priority |
|---|---|---|---|
| `n_estimators` | Number of trees | 100–1000 | Set it high enough and forget it |
| `max_features` | Features considered per split | `"sqrt"`, `"log2"`, 0.3–1.0 | **High** — the main decorrelation knob |
| `max_depth` | Tree height cap | `None`, or 5–30 | Medium — usually `None` is fine |
| `min_samples_leaf` | Minimum samples per leaf | 1–20 | **High** — the main smoothing knob |
| `min_samples_split` | Minimum to attempt a split | 2–20 | Medium |
| `class_weight` | Reweight classes | `None`, `"balanced"`, `"balanced_subsample"` | High if imbalanced |
| `bootstrap` | Sample rows with replacement | `True` | Leave it |
| `max_samples` | Fraction of rows per tree | 0.5–1.0 | Low; useful for speed on big data |

Random forests are famously **robust to default settings** — usually within a couple of
percent of a tuned model. Prefer `RandomizedSearchCV` over an exhaustive grid: the response
surface is flat, so random sampling finds a good region fast.

In [ ]:
# A realistic dataset with mixed signal strength and some useless columns
m2 = 3_000
X2 = pd.DataFrame({
    "tenure": rng.exponential(24, m2).clip(0, 90),
    "monthly": rng.normal(70, 22, m2).clip(15, 150),
    "support_calls": rng.poisson(1.4, m2),
    "n_services": rng.integers(1, 8, m2),
    "late_payments": rng.poisson(0.6, m2),
    "noise_a": rng.normal(size=m2),
    "noise_b": rng.integers(0, 100, m2),
    "noise_c": rng.random(m2),
})
z2 = (-1.1 - 0.05*X2.tenure + 0.017*X2.monthly + 0.32*X2.support_calls
      - 0.10*X2.n_services + 0.55*X2.late_payments
      + 0.6*((X2.monthly > 90) & (X2.tenure < 12)))          # a real interaction
y2 = (rng.random(m2) < 1/(1+np.exp(-z2))).astype(int)

Xa, Xb, ya, yb = train_test_split(X2, y2, test_size=0.25, random_state=0, stratify=y2)
print(f"{m2} customers, churn rate {y2.mean():.3f}, {X2.shape[1]} features "
      f"(3 of them pure noise)\n")

baseline = DummyClassifier(strategy="most_frequent").fit(Xa, ya)
tree = DecisionTreeClassifier(random_state=0).fit(Xa, ya)
tree_tuned = DecisionTreeClassifier(max_depth=5, min_samples_leaf=25, random_state=0).fit(Xa, ya)
forest = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xa, ya)
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xa, ya)

print(f"{'model':<28}{'train AUC':>11}{'CV AUC':>10}{'test AUC':>11}")
for name, mdl in [("baseline", baseline), ("single tree (unpruned)", tree),
                  ("single tree (tuned)", tree_tuned), ("logistic regression", logit),
                  ("random forest (defaults)", forest)]:
    try:
        tr = roc_auc_score(ya, mdl.predict_proba(Xa)[:, 1])
        te = roc_auc_score(yb, mdl.predict_proba(Xb)[:, 1])
        cvv = cross_val_score(mdl, Xa, ya, cv=SKF, scoring="roc_auc").mean()
    except Exception:
        tr = te = cvv = np.nan
    print(f"{name:<28}{tr:>11.4f}{cvv:>10.4f}{te:>11.4f}")
print("\nThe forest wins with zero tuning, and note its training AUC is ~1.0 while its")
print("test AUC holds up -- individual trees memorise, the ensemble does not.")

In [ ]:
# Randomised search over the parameters that matter
search = RandomizedSearchCV(
    RandomForestClassifier(n_estimators=300, random_state=0),
    param_distributions={
        "max_features": ["sqrt", "log2", 0.3, 0.5, 0.8, 1.0],
        "max_depth": [None, 5, 8, 12, 20],
        "min_samples_leaf": randint(1, 30),
        "min_samples_split": randint(2, 20),
    },
    n_iter=40, cv=SKF, scoring="roc_auc", random_state=0, n_jobs=1,
).fit(Xa, ya)

print(f"Best parameters : {search.best_params_}")
print(f"Best CV AUC     : {search.best_score_:.4f}")
print(f"Default CV AUC  : "
      f"{cross_val_score(RandomForestClassifier(n_estimators=300, random_state=0), Xa, ya, cv=SKF, scoring='roc_auc').mean():.4f}")
print(f"Tuned test AUC  : {roc_auc_score(yb, search.predict_proba(Xb)[:, 1]):.4f}")
print(f"Default test AUC: {roc_auc_score(yb, forest.predict_proba(Xb)[:, 1]):.4f}")
print("\nTuning bought a small improvement. That is typical for random forests, and it is")
print("why they are such a good first model: the default is close to the ceiling.")

In [ ]:
# Validation curves for the two knobs that matter most
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

leaf_range = [1, 2, 3, 5, 8, 12, 20, 35, 60, 100]
tr_l, va_l = validation_curve(RandomForestClassifier(n_estimators=150, random_state=0),
                              Xa, ya, param_name="min_samples_leaf",
                              param_range=leaf_range, cv=SKF, scoring="roc_auc")
ax[0].plot(leaf_range, tr_l.mean(1), "o-", color="steelblue", label="training")
ax[0].plot(leaf_range, va_l.mean(1), "o-", color="crimson", label="cross-validated")
ax[0].set_xscale("log"); ax[0].set_xlabel("min_samples_leaf")
ax[0].set_ylabel("ROC-AUC"); ax[0].legend(fontsize=8)
ax[0].set_title("min_samples_leaf smooths the forest")

mf_range = [1, 2, 3, 4, 5, 6, 7, 8]
tr_m, va_m = validation_curve(RandomForestClassifier(n_estimators=150, random_state=0),
                              Xa, ya, param_name="max_features",
                              param_range=mf_range, cv=SKF, scoring="roc_auc")
ax[1].plot(mf_range, tr_m.mean(1), "o-", color="steelblue", label="training")
ax[1].plot(mf_range, va_m.mean(1), "o-", color="crimson", label="cross-validated")
ax[1].axvline(np.sqrt(X2.shape[1]), color="black", ls="--",
              label=f"sqrt(p) = {np.sqrt(X2.shape[1]):.1f}")
ax[1].set_xlabel("max_features"); ax[1].legend(fontsize=8)
ax[1].set_title("max_features trades decorrelation against tree strength")
plt.tight_layout(); plt.show()

print(f"Best min_samples_leaf : {leaf_range[int(np.argmax(va_l.mean(1)))]}")
print(f"Best max_features     : {mf_range[int(np.argmax(va_m.mean(1)))]}")

---
## 6.5 Feature importance, done carefully

Random forests give you three ways to ask "which features matter?", and they disagree for
instructive reasons.

**1. Impurity-based (`feature_importances_`)** — mean impurity decrease across all trees.
Free, but biased toward **high-cardinality and continuous** features, and computed on the
**training** data.

**2. Permutation importance** — shuffle a column in held-out data and measure the drop.
Slower, honest, and measures what you care about. But with **correlated features** it can call
both unimportant, because shuffling one leaves the information available through the other.

**3. Drop-column importance** — refit without the column. The gold standard conceptually, and
expensive: one refit per feature.

Report permutation importance on a validation set, and be explicit about correlated groups.

In [ ]:
best_forest = search.best_estimator_

imp = pd.DataFrame({"feature": X2.columns,
                    "impurity": best_forest.feature_importances_})
perm = permutation_importance(best_forest, Xb, yb, n_repeats=30, random_state=0,
                             scoring="roc_auc")
imp["permutation"] = perm.importances_mean
imp["permutation_sd"] = perm.importances_std

# Drop-column importance
base_auc = cross_val_score(best_forest, Xa, ya, cv=SKF, scoring="roc_auc").mean()
drops = []
for col in X2.columns:
    sc = cross_val_score(RandomForestClassifier(n_estimators=150, random_state=0),
                         Xa.drop(columns=col), ya, cv=SKF, scoring="roc_auc").mean()
    drops.append(base_auc - sc)
imp["drop_column"] = drops

print(imp.sort_values("permutation", ascending=False).round(4).to_string(index=False))
print(f"\n(noise_a, noise_b and noise_c carry NO signal by construction)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
for a_, col, title in zip(ax, ["impurity", "permutation", "drop_column"],
                          ["Impurity-based (training, biased)",
                           "Permutation (test set)",
                           "Drop-column (refit per feature)"]):
    o = imp.sort_values(col)
    colours = ["crimson" if "noise" in f else "steelblue" for f in o.feature]
    a_.barh(o.feature, o[col], color=colours)
    a_.axvline(0, color="black", lw=0.8)
    a_.set_title(title, fontsize=10)
plt.tight_layout(); plt.show()

print("Red bars are the pure-noise features.")
print("Impurity importance gives noise_b and noise_c non-trivial credit (noise_b has 100")
print("distinct values, noise_c has 2,250 -- both offer many candidate thresholds).")
print("Permutation and drop-column importance correctly put them near zero.")

In [ ]:
# The correlated-feature trap
X3 = X2.copy()
X3["monthly_copy"] = X2.monthly + rng.normal(0, 0.5, m2)     # near-duplicate of a key feature
Xa3, Xb3, ya3, yb3 = train_test_split(X3, y2, test_size=0.25, random_state=0, stratify=y2)
f3 = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xa3, ya3)
p3 = permutation_importance(f3, Xb3, yb3, n_repeats=25, random_state=0, scoring="roc_auc")

comp = pd.DataFrame({"feature": X3.columns, "permutation": p3.importances_mean}
                    ).sort_values("permutation", ascending=False)
print(comp.round(4).to_string(index=False))
print(f"\nWithout the duplicate, 'monthly' had permutation importance "
      f"{imp.loc[imp.feature == 'monthly', 'permutation'].item():.4f}.")
print(f"With a near-duplicate present, it drops to "
      f"{comp.loc[comp.feature == 'monthly', 'permutation'].item():.4f} -- and so does the copy.")
print("Neither looks important, because shuffling one leaves the other to fill in.")
print("\nFix: permute correlated features as a GROUP, or cluster features first and")
print("report importance per cluster.")

group = ["monthly", "monthly_copy"]
Xb3_shuf = Xb3.copy()
for c in group:
    Xb3_shuf[c] = rng.permutation(Xb3_shuf[c].to_numpy())
drop_group = (roc_auc_score(yb3, f3.predict_proba(Xb3)[:, 1])
              - roc_auc_score(yb3, f3.predict_proba(Xb3_shuf)[:, 1]))
print(f"\nPermuting BOTH monthly columns together: AUC drop = {drop_group:.4f}")
print("Now the true importance of 'monthly information' is visible again.")

In [ ]:
# Partial dependence: not just WHICH features matter, but HOW
fig, ax = plt.subplots(figsize=(13, 4))
PartialDependenceDisplay.from_estimator(
    best_forest, Xa, features=["tenure", "monthly", "late_payments"],
    ax=ax, line_kw={"color": "crimson", "lw": 2})
plt.suptitle("Partial dependence: the average effect of each feature on churn probability",
             y=1.03, fontsize=11)
plt.tight_layout(); plt.show()

print("Reading these: churn probability falls steeply over the first ~20 months of tenure")
print("then flattens; it rises with monthly charges and with late payments. The forest")
print("discovered these shapes without being told they were monotone.")
print("\nCaveat: partial dependence averages over the other features, so it hides")
print("interactions. For a 2-D view, pass a tuple of two features.")

In [ ]:
# Two-way partial dependence recovers the interaction we planted
fig, ax = plt.subplots(figsize=(6, 5))
PartialDependenceDisplay.from_estimator(
    best_forest, Xa, features=[("tenure", "monthly")], ax=ax)
plt.title("Interaction: high charges hurt most for NEW customers", fontsize=10)
plt.tight_layout(); plt.show()

print("The data was generated with a term that only fires when monthly > 90 AND tenure < 12.")
print("The contour plot shows exactly that corner -- an interaction a linear model would")
print("have needed an explicit product term to see.")

---
## 6.6 The neighbours: Extra Trees and boosting

| Method | How trees are built | Character |
|---|---|---|
| **Bagging** | Bootstrap rows, all features per split | Reduces variance |
| **Random Forest** | Bootstrap rows + random feature subset per split | Reduces variance more |
| **Extra Trees** | *All* rows (no bootstrap by default) + random feature subset + **random thresholds** | Even more decorrelated; faster; slightly more bias |
| **Gradient Boosting** | Trees built **sequentially**, each fitting the previous residuals | Reduces **bias**; needs tuning and early stopping |

The distinction that matters: **forests are parallel and variance-reducing; boosting is
sequential and bias-reducing.** A forest of 500 trees cannot overfit by adding tree 501. A
boosting model absolutely can.

For tabular competitions, gradient boosting (XGBoost, LightGBM, CatBoost, or scikit-learn's
`HistGradientBoosting`) usually wins — at the cost of much more careful tuning.

In [ ]:
candidates = {
    "single tree (tuned)": DecisionTreeClassifier(max_depth=6, min_samples_leaf=20,
                                                 random_state=0),
    "bagging (300)": BaggingClassifier(DecisionTreeClassifier(), n_estimators=300,
                                       random_state=0),
    "random forest (300)": RandomForestClassifier(n_estimators=300, random_state=0),
    "extra trees (300)": ExtraTreesClassifier(n_estimators=300, random_state=0),
    "gradient boosting": GradientBoostingClassifier(random_state=0),
    "hist gradient boosting": HistGradientBoostingClassifier(random_state=0),
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
}
rows_c = []
for name, est in candidates.items():
    t0 = time.perf_counter()
    cvv = cross_val_score(est, Xa, ya, cv=SKF, scoring="roc_auc")
    t1 = time.perf_counter()
    est.fit(Xa, ya)
    rows_c.append({"model": name, "CV_AUC": cvv.mean(), "CV_sd": cvv.std(),
                   "test_AUC": roc_auc_score(yb, est.predict_proba(Xb)[:, 1]),
                   "fit_time_s": round((t1-t0)/5, 3)})
comp = pd.DataFrame(rows_c).sort_values("CV_AUC", ascending=False)
print(comp.round(4).to_string(index=False))
print("\nAll the ensembles beat the single tree comfortably. On this dataset the boosting")
print("models edge out the forest, as they usually do -- but they came with defaults that")
print("happened to work; tuning boosting properly takes far longer than tuning a forest.")

In [ ]:
# The key behavioural difference: boosting CAN overfit as you add trees, a forest cannot
n_range = [5, 10, 25, 50, 100, 200, 400, 800]
rf_scores, gb_scores = [], []
for B in n_range:
    rf = RandomForestClassifier(n_estimators=B, random_state=0).fit(Xa, ya)
    gb = GradientBoostingClassifier(n_estimators=B, learning_rate=0.2, max_depth=5,
                                    random_state=0).fit(Xa, ya)
    rf_scores.append(roc_auc_score(yb, rf.predict_proba(Xb)[:, 1]))
    gb_scores.append(roc_auc_score(yb, gb.predict_proba(Xb)[:, 1]))

plt.plot(n_range, rf_scores, "o-", color="steelblue", label="random forest")
plt.plot(n_range, gb_scores, "o-", color="crimson", label="gradient boosting (lr=0.2, depth=5)")
plt.xscale("log"); plt.xlabel("number of trees"); plt.ylabel("test ROC-AUC")
plt.title("Forests plateau; aggressive boosting eventually degrades")
plt.legend(fontsize=8); plt.show()

print(f"Random forest    : best {max(rf_scores):.4f} at {n_range[int(np.argmax(rf_scores))]} "
      f"trees, at 800 trees {rf_scores[-1]:.4f}")
print(f"Gradient boosting: best {max(gb_scores):.4f} at {n_range[int(np.argmax(gb_scores))]} "
      f"trees, at 800 trees {gb_scores[-1]:.4f}")
print("\nThis is why boosting needs early stopping and a forest does not.")

---
## 6.7 Random forests for regression

Identical machinery: each tree predicts the mean of its leaf, and the forest averages those
predictions. The impurity criterion is squared error.

The averaging smooths the staircase of a single tree into something much closer to a curve —
but the **inability to extrapolate is inherited**. A forest of flat-outside-the-range trees is
still flat outside the range.

In [ ]:
xs = np.sort(rng.uniform(0, 10, 200))
ys = np.sin(xs) + 0.3*xs + rng.normal(0, 0.35, 200)
Xr = xs.reshape(-1, 1)
grid = np.linspace(-2, 15, 700).reshape(-1, 1)

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a_, (mdl, name) in zip(ax, [
        (DecisionTreeRegressor(max_depth=4, random_state=0), "single tree, depth 4"),
        (RandomForestRegressor(n_estimators=300, random_state=0), "random forest, 300 trees"),
        (RandomForestRegressor(n_estimators=300, min_samples_leaf=10, random_state=0),
         "forest, min_samples_leaf=10")]):
    mdl.fit(Xr, ys)
    a_.scatter(xs, ys, s=12, alpha=0.5, color="steelblue")
    a_.plot(grid, mdl.predict(grid), color="crimson", lw=2)
    a_.axvspan(-2, 0, color="grey", alpha=0.12); a_.axvspan(10, 15, color="grey", alpha=0.12)
    cvv = -cross_val_score(mdl, Xr, ys, cv=CV, scoring="neg_root_mean_squared_error").mean()
    a_.set_title(f"{name}\nCV RMSE {cvv:.4f}", fontsize=9)
plt.tight_layout(); plt.show()

print("The forest's prediction is much smoother than a single tree's -- averaging 300")
print("staircases with different step positions produces something nearly continuous.")
print("But look at the grey zones: still perfectly flat. Averaging does not create the")
print("ability to extrapolate.")

In [ ]:
# Prediction intervals from the spread across trees -- a genuinely useful trick
rf_reg = RandomForestRegressor(n_estimators=400, min_samples_leaf=5, random_state=0).fit(Xr, ys)
per_tree = np.array([t.predict(grid) for t in rf_reg.estimators_])
mean_pred = per_tree.mean(axis=0)
lo, hi = np.percentile(per_tree, [5, 95], axis=0)

plt.scatter(xs, ys, s=12, alpha=0.5, color="steelblue", label="data")
plt.plot(grid, mean_pred, color="crimson", lw=2, label="forest mean")
plt.fill_between(grid.ravel(), lo, hi, color="crimson", alpha=0.2,
                 label="5th-95th percentile across trees")
plt.xlim(0, 10); plt.legend(fontsize=8)
plt.title("Tree-to-tree spread as an uncertainty estimate")
plt.show()

print("Where the trees disagree, the model is uncertain -- typically in sparse regions and")
print("near sharp changes. This is a cheap and very practical uncertainty signal.")
print("\nA caution: this spread measures MODEL uncertainty (variance of the estimator),")
print("not the irreducible noise in y. It is narrower than a true prediction interval.")
print("For calibrated intervals use quantile regression forests or conformal prediction.")

In [ ]:
# Regression on real data
from sklearn.datasets import load_diabetes
dia = load_diabetes(as_frame=True)
Xd, yd = dia.data, dia.target
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.25, random_state=0)

from sklearn.linear_model import RidgeCV
for name, est in [("Ridge", make_pipeline(StandardScaler(), RidgeCV())),
                  ("single tree (depth 4)", DecisionTreeRegressor(max_depth=4, random_state=0)),
                  ("random forest", RandomForestRegressor(n_estimators=400, random_state=0)),
                  ("random forest (leaf=5)",
                   RandomForestRegressor(n_estimators=400, min_samples_leaf=5, random_state=0)),
                  ("hist gradient boosting", HistGradientBoostingRegressor(random_state=0))]:
    cvv = -cross_val_score(est, Xd_tr, yd_tr, cv=CV,
                           scoring="neg_root_mean_squared_error").mean()
    est.fit(Xd_tr, yd_tr)
    te = np.sqrt(mean_squared_error(yd_te, est.predict(Xd_te)))
    print(f"  {name:<24} CV RMSE {cvv:>7.2f}   test RMSE {te:>7.2f}   "
          f"test R2 {r2_score(yd_te, est.predict(Xd_te)):>6.3f}")
print("\nOn this small, noisy, roughly linear dataset, regularised linear regression is")
print("competitive with the forest. Forests are not automatically better -- they win when")
print("there are interactions and non-linearities to find.")

---
## 6.8 Imbalanced classes

Three options, and they are not equivalent:

- **`class_weight="balanced"`** — weight each class inversely to its frequency, computed once
  on the whole training set
- **`class_weight="balanced_subsample"`** — recompute the weights per bootstrap sample. Usually
  the better choice for a forest
- **Threshold the probabilities** — leave the model alone and pick an operating point, as in
  Notebook 2

As always: judge with PR-AUC and recall, not accuracy.

In [ ]:
m_i = 8_000
Xi = pd.DataFrame(rng.normal(size=(m_i, 8)),
                  columns=[f"f{j}" for j in range(8)])
zi = -3.7 + 1.3*Xi.f0 + 0.9*Xi.f1 - 0.7*Xi.f2 + 0.6*Xi.f0*Xi.f1
yi = (rng.random(m_i) < 1/(1+np.exp(-zi))).astype(int)
Xa4, Xb4, ya4, yb4 = train_test_split(Xi, yi, test_size=0.3, random_state=0, stratify=yi)
print(f"Positive rate: {yi.mean():.4f}\n")

rows_i = []
for name, est in [
        ("baseline (majority)", DummyClassifier(strategy="most_frequent")),
        ("forest, no weighting", RandomForestClassifier(n_estimators=300, random_state=0)),
        ("forest, balanced", RandomForestClassifier(n_estimators=300,
                                                    class_weight="balanced", random_state=0)),
        ("forest, balanced_subsample",
         RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                random_state=0))]:
    est.fit(Xa4, ya4)
    pr = est.predict(Xb4)
    try:
        pb = est.predict_proba(Xb4)[:, 1]
        auc, ap = roc_auc_score(yb4, pb), average_precision_score(yb4, pb)
    except Exception:
        auc = ap = np.nan
    rows_i.append({"model": name, "accuracy": accuracy_score(yb4, pr),
                   "precision": precision_score(yb4, pr, zero_division=0),
                   "recall": recall_score(yb4, pr, zero_division=0),
                   "ROC_AUC": auc, "PR_AUC": ap})
print(pd.DataFrame(rows_i).round(4).to_string(index=False))
print("\nNote that ROC-AUC and PR-AUC barely change: the RANKING of customers is the same.")
print("Class weighting moves the implicit threshold, nothing more. So you may as well")
print("move the threshold explicitly and keep the probabilities interpretable.")

In [ ]:
# Threshold tuning on the unweighted forest achieves the same recall, with control
f_plain = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xa4, ya4)
pb4 = f_plain.predict_proba(Xb4)[:, 1]
target_recall = 0.70
ths = np.linspace(0.001, 0.6, 600)
ok = [(t, precision_score(yb4, (pb4 >= t).astype(int), zero_division=0))
      for t in ths if recall_score(yb4, (pb4 >= t).astype(int)) >= target_recall]
t_star, p_star = max(ok, key=lambda z: z[1])
print(f"To reach recall >= {target_recall}, use threshold {t_star:.4f}")
print(f"  precision {p_star:.4f}, recall "
      f"{recall_score(yb4, (pb4 >= t_star).astype(int)):.4f}")
print(f"  flagged {int((pb4 >= t_star).sum())} of {len(pb4)} customers")
print("\nOne model, any operating point you like, and the probabilities still mean")
print("something. This is why threshold tuning is the first tool to reach for.")

---
## 6.9 Strengths, weaknesses, and when to use one

**Strengths**

- **Excellent out of the box** — usually within a few percent of a tuned model
- Handles non-linearities and interactions automatically
- No scaling needed
- Robust to outliers and to irrelevant features
- **Free validation** via OOB
- Gives feature importances and an uncertainty estimate
- Parallelises trivially (`n_jobs=-1`)
- Very hard to overfit by adding trees

**Weaknesses**

- **Not interpretable** as a whole — 300 trees is not a flowchart
- **Cannot extrapolate**
- **Large models** — hundreds of trees to store and traverse; slow inference relative to a
  linear model
- Biased impurity importances
- Struggles with **very high-dimensional sparse** data (text), where linear models win
- Usually beaten by well-tuned gradient boosting on tabular problems

**Reach for a random forest when** you have a tabular dataset, a deadline, and you want a
strong, low-risk baseline that will tell you what accuracy is achievable.

In [ ]:
# The interpretability/accuracy frontier on our churn data
frontier = {
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "tree, depth 3": DecisionTreeClassifier(max_depth=3, min_samples_leaf=30, random_state=0),
    "tree, depth 6": DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, random_state=0),
    "random forest": RandomForestClassifier(n_estimators=300, random_state=0),
    "hist gradient boosting": HistGradientBoostingClassifier(random_state=0),
}
explain = {"logistic regression": "coefficients / odds ratios",
           "tree, depth 3": "readable rulebook (<=3 questions)",
           "tree, depth 6": "rulebook, but ~40 leaves",
           "random forest": "importances + partial dependence only",
           "hist gradient boosting": "importances + SHAP only"}
sizes = {}
for name, est in frontier.items():
    cvv = cross_val_score(est, Xa, ya, cv=SKF, scoring="roc_auc")
    est.fit(Xa, ya)
    print(f"  {name:<24} CV AUC {cvv.mean():.4f} +/- {cvv.std():.4f}   "
          f"explanation: {explain[name]}")
print("\nThe forest buys accuracy with explainability. Whether that is the right trade")
print("depends on whether a human has to defend each decision -- see Notebook 3,")
print("Exercise 4 for how to argue that case with numbers.")

---
## Exercises

**Exercise 1.** On the breast cancer dataset, compare a single tree, a random forest and extra
trees. Report CV accuracy and ROC-AUC, and show that the forest's OOB score approximates its
CV score.

In [ ]:
# --- Solution 1 -------------------------------------------------------------
from sklearn.datasets import load_breast_cancer

bc = load_breast_cancer(as_frame=True)
Xbc, ybc = bc.data, bc.target
Xbc_tr, Xbc_te, ybc_tr, ybc_te = train_test_split(Xbc, ybc, test_size=0.25, random_state=0,
                                                  stratify=ybc)
print(f"{Xbc.shape[0]} samples, {Xbc.shape[1]} features, positive rate {ybc.mean():.3f}\n")

print(f"{'model':<26}{'CV acc':>9}{'CV AUC':>9}{'OOB':>9}{'test AUC':>11}")
for name, est in [("single tree", DecisionTreeClassifier(random_state=0)),
                  ("random forest (300)", RandomForestClassifier(n_estimators=300,
                                                                 oob_score=True,
                                                                 random_state=0)),
                  ("extra trees (300)", ExtraTreesClassifier(n_estimators=300,
                                                             bootstrap=True, oob_score=True,
                                                             random_state=0))]:
    cv_acc = cross_val_score(est, Xbc_tr, ybc_tr, cv=SKF).mean()
    cv_auc = cross_val_score(est, Xbc_tr, ybc_tr, cv=SKF, scoring="roc_auc").mean()
    est.fit(Xbc_tr, ybc_tr)
    oob_ = getattr(est, "oob_score_", np.nan)
    te = roc_auc_score(ybc_te, est.predict_proba(Xbc_te)[:, 1])
    print(f"{name:<26}{cv_acc:>9.4f}{cv_auc:>9.4f}{oob_:>9.4f}{te:>11.4f}")

print("\nThe OOB score sits within a whisker of the CV accuracy, at one fifth of the cost.")
print("Extra trees match or beat the forest here and fit faster, because random thresholds")
print("skip the exhaustive split search.")

**Exercise 2.** Show the impurity-importance bias on real data: add a random continuous column
and a random 100-level categorical column to the breast cancer features, and compare impurity
against permutation importance.

In [ ]:
# --- Solution 2 -------------------------------------------------------------
Xbc2 = Xbc.copy()
Xbc2["random_continuous"] = rng.random(len(Xbc2))
Xbc2["random_100_levels"] = rng.integers(0, 100, len(Xbc2))
Xbc2["random_binary"] = rng.integers(0, 2, len(Xbc2))

Xa5, Xb5, ya5, yb5 = train_test_split(Xbc2, ybc, test_size=0.3, random_state=0, stratify=ybc)
f5 = RandomForestClassifier(n_estimators=400, random_state=0).fit(Xa5, ya5)
p5 = permutation_importance(f5, Xb5, yb5, n_repeats=25, random_state=0, scoring="roc_auc")

comp5 = pd.DataFrame({"feature": Xbc2.columns,
                      "impurity": f5.feature_importances_,
                      "permutation": p5.importances_mean})
fakes = ["random_continuous", "random_100_levels", "random_binary"]

print("The three injected NOISE features:")
print(comp5[comp5.feature.isin(fakes)].round(5).to_string(index=False))
print(f"\nTheir impurity ranks out of {len(comp5)} features "
      f"(1 = most important):")
comp5["impurity_rank"] = comp5.impurity.rank(ascending=False).astype(int)
comp5["permutation_rank"] = comp5.permutation.rank(ascending=False).astype(int)
print(comp5[comp5.feature.isin(fakes)][["feature", "impurity_rank",
                                        "permutation_rank"]].to_string(index=False))
print(f"\nTotal impurity importance assigned to pure noise: "
      f"{comp5[comp5.feature.isin(fakes)].impurity.sum():.4f} "
      f"({comp5[comp5.feature.isin(fakes)].impurity.sum()*100:.1f}% of the total)")
print(f"Total permutation importance for the same columns: "
      f"{comp5[comp5.feature.isin(fakes)].permutation.sum():.4f}")
print("\nNotice the ordering within the fakes: the continuous column and the 100-level")
print("column get far more impurity credit than the binary one, purely because they offer")
print("more candidate split points. That is the cardinality bias in one table.")

**Exercise 3.** Tune a random forest for a business goal rather than for accuracy: a fraud team
can review 300 cases per day out of 50,000 transactions. Maximise the number of frauds caught
in the top 300 (precision@300).

In [ ]:
# --- Solution 3 -------------------------------------------------------------
m_f = 50_000
Xfr = pd.DataFrame({
    "amount": rng.lognormal(4.2, 1.1, m_f),
    "n_countries_24h": rng.poisson(0.5, m_f),
    "hour": rng.integers(0, 24, m_f),
    "device_age_days": rng.exponential(200, m_f),
    "velocity_1h": rng.poisson(1.2, m_f),
})
zf = (-6.4 + 0.42*np.log(Xfr.amount) + 0.95*Xfr.n_countries_24h
      + 0.55*((Xfr.hour < 5) | (Xfr.hour > 22)).astype(int)
      - 0.004*Xfr.device_age_days + 0.35*Xfr.velocity_1h)
yfr = (rng.random(m_f) < 1/(1+np.exp(-zf))).astype(int)
Xa6, Xb6, ya6, yb6 = train_test_split(Xfr, yfr, test_size=0.3, random_state=0, stratify=yfr)
print(f"Fraud rate: {yfr.mean():.4f} ({yfr.sum()} frauds)")

CAPACITY = int(300 * len(yb6) / m_f)     # review capacity scaled to the test set
print(f"Review capacity on the test set: {CAPACITY} cases\n")

def precision_at_k(y_true, scores, k):
    top = np.argsort(scores)[::-1][:k]
    return np.asarray(y_true)[top].sum() / k

def frauds_caught_at_k(y_true, scores, k):
    top = np.argsort(scores)[::-1][:k]
    return int(np.asarray(y_true)[top].sum())

scorer = lambda est, X, y: precision_at_k(y, est.predict_proba(X)[:, 1], CAPACITY)

search3 = RandomizedSearchCV(
    RandomForestClassifier(n_estimators=200, random_state=0),
    {"max_features": ["sqrt", 0.5, 1.0], "min_samples_leaf": randint(1, 40),
     "max_depth": [None, 8, 14], "class_weight": [None, "balanced_subsample"]},
    n_iter=18, cv=SKF, scoring=scorer, random_state=0, n_jobs=1,
).fit(Xa6, ya6)

print(f"Best parameters (optimising precision@{CAPACITY}): {search3.best_params_}")
print(f"Best CV precision@k : {search3.best_score_:.4f}\n")

In [ ]:
plain6 = RandomForestClassifier(n_estimators=200, random_state=0).fit(Xa6, ya6)
logit6 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xa6, ya6)

print(f"{'model':<34}{'precision@k':>13}{'frauds caught':>15}{'ROC_AUC':>10}{'PR_AUC':>9}")
for name, est in [("random access (random order)", None),
                  ("logistic regression", logit6),
                  ("random forest, defaults", plain6),
                  ("random forest, tuned for p@k", search3.best_estimator_)]:
    if est is None:
        sc = rng.random(len(yb6))
    else:
        sc = est.predict_proba(Xb6)[:, 1]
    print(f"{name:<34}{precision_at_k(yb6, sc, CAPACITY):>13.4f}"
          f"{frauds_caught_at_k(yb6, sc, CAPACITY):>15}"
          f"{roc_auc_score(yb6, sc):>10.4f}{average_precision_score(yb6, sc):>9.4f}")

best_sc = search3.best_estimator_.predict_proba(Xb6)[:, 1]
lift = precision_at_k(yb6, best_sc, CAPACITY) / yb6.mean()
print(f"\nLift over reviewing cases at random: {lift:.1f}x")
print(f"Total frauds in the test set: {yb6.sum()}, of which we catch "
      f"{frauds_caught_at_k(yb6, best_sc, CAPACITY)} "
      f"({frauds_caught_at_k(yb6, best_sc, CAPACITY)/yb6.sum():.1%}) using only "
      f"{CAPACITY} reviews.")
print("\nThe point of the exercise: the model was selected by the metric the BUSINESS")
print("cares about (how many frauds fit in a fixed review queue), not by accuracy or")
print("even AUC. A model with slightly worse AUC can be better at the top of the ranking,")
print("and the top of the ranking is all the fraud team ever sees.")

**Exercise 4 (challenge).** Build a complete model-selection study on one dataset: baseline,
linear model, single tree, forest, extra trees and boosting. Include learning curves, an
honest test-set evaluation, timing, and a recommendation with the reasoning made explicit.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
X7, y7 = Xa, ya                      # the churn training data from section 6.4
X7_te, y7_te = Xb, yb

models = {
    "baseline (majority)": DummyClassifier(strategy="most_frequent"),
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "decision tree (tuned)": DecisionTreeClassifier(max_depth=6, min_samples_leaf=20,
                                                    random_state=0),
    "random forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=3,
                                            random_state=0),
    "extra trees": ExtraTreesClassifier(n_estimators=400, min_samples_leaf=3, random_state=0),
    "hist gradient boosting": HistGradientBoostingClassifier(random_state=0),
}
rows7 = []
for name, est in models.items():
    t0 = time.perf_counter()
    cvv = cross_val_score(est, X7, y7, cv=SKF, scoring="roc_auc")
    fit_s = (time.perf_counter() - t0) / 5
    est.fit(X7, y7)
    t1 = time.perf_counter(); est.predict(X7_te); pred_ms = (time.perf_counter()-t1)*1000
    try:
        pb = est.predict_proba(X7_te)[:, 1]
        te_auc, te_ap = roc_auc_score(y7_te, pb), average_precision_score(y7_te, pb)
    except Exception:
        te_auc = te_ap = np.nan
    rows7.append({"model": name, "CV_AUC": cvv.mean(), "CV_sd": cvv.std(),
                  "test_AUC": te_auc, "test_PR_AUC": te_ap,
                  "fit_s": round(fit_s, 3), "predict_ms": round(pred_ms, 1)})
study = pd.DataFrame(rows7).sort_values("CV_AUC", ascending=False)
print(study.round(4).to_string(index=False))

In [ ]:
# Learning curves for the three serious contenders
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, est) in zip(axes, [
        ("logistic regression", models["logistic regression"]),
        ("random forest", models["random forest"]),
        ("hist gradient boosting", models["hist gradient boosting"])]):
    sizes, tr, va = learning_curve(est, X7, y7, train_sizes=np.linspace(0.1, 1.0, 8),
                                   cv=SKF, scoring="roc_auc")
    ax.plot(sizes, tr.mean(1), "o-", color="steelblue", label="training")
    ax.plot(sizes, va.mean(1), "o-", color="crimson", label="cross-validated")
    ax.fill_between(sizes, va.mean(1)-va.std(1), va.mean(1)+va.std(1),
                    color="crimson", alpha=0.15)
    ax.set_xlabel("training rows"); ax.set_ylabel("ROC-AUC")
    ax.set_title(f"{name}\ngap at full size: {tr.mean(1)[-1]-va.mean(1)[-1]:.3f}", fontsize=9)
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print("Reading these: the linear model's curves meet low (bias-limited); the forest has a")
print("large but harmless gap (it memorises trees, the ensemble still generalises); the CV")
print("curve is still rising slightly for the ensembles, so more data would help a little.")

In [ ]:
best_name = study.iloc[0]["model"]
best_est = models[best_name].fit(X7, y7)
pb7 = best_est.predict_proba(X7_te)[:, 1]

print("=" * 74)
print(f"RECOMMENDATION")
print("=" * 74)
print(f"Best CV model : {best_name} (CV AUC {study.iloc[0].CV_AUC:.4f} "
      f"+/- {study.iloc[0].CV_sd:.4f})")
print(f"Test AUC      : {study.iloc[0].test_AUC:.4f}   test PR-AUC "
      f"{study.iloc[0].test_PR_AUC:.4f}")
print(f"vs baseline   : {study[study.model == 'baseline (majority)'].test_AUC.item():.4f}")
print(f"vs linear     : {study[study.model == 'logistic regression'].test_AUC.item():.4f}")
one_se = study.CV_AUC.max() - study.CV_sd[study.CV_AUC.idxmax()]
within = study[study.CV_AUC >= one_se]["model"].tolist()
print(f"\nOne-standard-error rule: models statistically tied with the best (CV AUC >= "
      f"{one_se:.4f}):")
print(f"  {within}")
print("  If one of those is materially simpler or cheaper, prefer it over the raw maximum.")
print()
print("REASONING")
print("  1. The ensembles beat logistic regression by a clear margin, which tells us the")
print("     data really does contain interactions -- consistent with how it was generated.")
print("  2. Forest and boosting are within a standard error of each other. The forest is")
print("     the safer engineering choice: fewer hyperparameters, cannot overfit by adding")
print("     trees, parallel training, and an OOB estimate for monitoring.")
print(f"  3. Inference cost: {study[study.model=='random forest'].predict_ms.item():.0f} ms "
      f"for {len(X7_te)} rows, versus "
      f"{study[study.model=='logistic regression'].predict_ms.item():.0f} ms for the linear")
print("     model. Fine for batch scoring; check it against your latency budget for")
print("     real-time use.")
print("  4. Explainability: deliver permutation importances and partial-dependence plots")
print("     with the model, and keep the depth-6 tree as a human-readable approximation.")
print()
print("WHAT I WOULD DO NEXT")
print("  * tune the decision threshold to the retention team's actual capacity and costs")
print("  * check calibration; forests are usually slightly over-confident near 0 and 1")
print("  * re-validate on a TIME-ORDERED split before deployment, since churn is temporal")
print("  * monitor feature distributions and the OOB score for drift after launch")
print("=" * 74)

---
## Summary

| Concept | Key point |
|---|---|
| Bagging | Bootstrap samples + averaging; reduces **variance**, not bias |
| Variance of an average | $\rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$ — decorrelation is the real lever |
| Random forest | Bagging **plus** a random feature subset at each split |
| `max_features` | The decorrelation knob; $\sqrt{p}$ for classification |
| `n_estimators` | More is never worse; 100–500 is plenty |
| OOB score | Free validation from the ~37% of rows each tree did not see |
| `min_samples_leaf` | The main smoothing knob |
| Impurity importance | Biased toward high-cardinality features; training-based |
| Permutation importance | Honest, but splits credit oddly among correlated features |
| Partial dependence | Shows *how* a feature acts, not just that it matters |
| Extra Trees | Random thresholds too; faster, more decorrelated |
| Boosting | Sequential, bias-reducing, **can** overfit with more trees |
| Regression | Smooth staircases, tree spread gives uncertainty, still no extrapolation |
| Best use | Strong, low-risk default for any tabular problem |

**Next up:** [Notebook 7 — Support Vector Machines](7.%20Support%20Vector%20Machines.ipynb),
which takes the opposite approach: one boundary, chosen to be as far from the data as possible.